# Diffusion LoRA SDXL para Campañas Automotrices

Notebook enfocado únicamente en el fine-tuning visual del modelo de difusión. Entrena un LoRA SDXL con Diffusers, guarda artefactos en Google Drive y compara el modelo base contra el modelo fine-tuned con métricas reproducibles.

Fuentes esperadas dentro del repo:

```text
data/car_campaign_lora/metadata.csv
data/car_campaign_lora/metadata_template.csv
data/car_campaign_lora/images/
```

El modo por defecto es `smoke_free`: usa pocas imágenes y pocos pasos para validar el pipeline en Colab Free. Cambia a `pro` cuando uses Colab Pro y quieras entrenar con más pasos/checkpoints.

## 0. Runtime y GPU

Activa GPU antes de entrenar. En Colab: `Runtime > Change runtime type > T4/A100/L4 GPU`. Esta celda solo diagnostica el entorno.

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("GPU detectada:")
    print("\n".join(result.stdout.split("\n")[:18]))
else:
    print("No se detecto GPU. Puedes validar datos, pero activa GPU para entrenar/generar con SDXL.")

## 1. Instalar dependencias

La instalación está controlada por flags para poder reejecutar el notebook sin reinstalar todo. Si Colab ya importó `torch`, `PIL` o `torchvision` antes de instalar, reinicia el runtime una vez después de esta celda.

In [ ]:
INSTALL_DEPENDENCIES = True
INSTALL_DIFFUSERS_FROM_GITHUB = True
INSTALL_EVAL_DEPENDENCIES = True
REMOVE_INCOMPATIBLE_TORCHAO = True
PYTORCH_CUDA_INDEX_URL = "https://download.pytorch.org/whl/cu128"

if INSTALL_DEPENDENCIES:
    !pip install -q --upgrade pip
    !pip install -q --upgrade --no-cache-dir \
        "jedi>=0.16" \
        "numpy>=1.26,<2.1" \
        "pandas==2.2.2" \
        "Pillow>=10.4.0,<12.0" \
        "requests==2.32.4" \
        "huggingface_hub>=0.25.0" \
        "accelerate>=1.1.0" \
        "peft>=0.13.0" \
        "safetensors>=0.4.5" \
        "transformers>=4.46.0" \
        "matplotlib>=3.8.0"

    # Mantiene torch/torchvision en la misma familia CUDA para evitar errores de runtime.
    !pip install -q --upgrade --force-reinstall --no-cache-dir \
        --index-url {PYTORCH_CUDA_INDEX_URL} \
        "torch==2.11.0" "torchvision==0.26.0" "torchaudio==2.11.0"

    if INSTALL_DIFFUSERS_FROM_GITHUB:
        !pip install -q --no-deps git+https://github.com/huggingface/diffusers.git
    else:
        !pip install -q --upgrade --no-deps diffusers

    if REMOVE_INCOMPATIBLE_TORCHAO:
        # En Colab aparece a veces torchao preinstalado en una version vieja que rompe PEFT/LoRA.
        !pip uninstall -y torchao >/dev/null 2>&1 || true

    if INSTALL_EVAL_DEPENDENCIES:
        # torchmetrics se usa solo si esta disponible; el notebook tambien tiene fallback manual para CLIP.
        !pip install -q --upgrade "torchmetrics>=1.6.0"

print("Dependencias listas. Si acabas de instalarlas en Colab, reinicia el runtime una vez y continua desde la siguiente celda.")
print("Nota: si Colab traia torchao viejo, se desinstalo para evitar el error de PEFT al inyectar LoRA.")

## 2. Imports, semilla y Google Drive

Todo lo que pesa o debe sobrevivir a la sesión se guarda en Drive bajo `DRIVE_PROJECT_DIR`.

In [ ]:
import gc
import importlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image, ImageOps

try:
    import torchvision.transforms  # noqa: F401
except ModuleNotFoundError:
    print("torchvision no esta instalado; algunas dependencias pueden pedirlo luego.")
except (ImportError, RuntimeError) as exc:
    raise RuntimeError(
        "torchvision quedo incompatible con torch/CUDA. Ejecuta la celda de instalacion, "
        "reinicia el runtime y vuelve a correr desde aqui. "
        f"torch={getattr(torch, '__version__', 'no disponible')}, "
        f"cuda={getattr(torch.version, 'cuda', 'no disponible')}. Error original: {exc}"
    )

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("No parece ser Colab. Se usaran rutas locales para validacion.")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/dmc-tp3-gen-multimodal/diffusion_lora") if IN_COLAB else Path.cwd() / "outputs" / "diffusion_lora_drive_mock"
DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)

def get_installed_package_version(package_name: str):
    try:
        import importlib.metadata as importlib_metadata
        return importlib_metadata.version(package_name)
    except Exception:
        return None

for pkg in ["torch", "torchvision", "transformers", "diffusers", "accelerate", "peft", "torchmetrics"]:
    try:
        module = importlib.import_module(pkg)
        print(pkg, getattr(module, "__version__", "version no disponible"))
    except Exception as exc:
        print(pkg, "no disponible:", exc)

torchao_version = get_installed_package_version("torchao")
print("torchao:", torchao_version or "no instalado")
if torchao_version and tuple(int(part) for part in torchao_version.split(".")[:2]) < (0, 16):
    raise RuntimeError(
        "Se detecto torchao incompatible (" + torchao_version + "). "
        "Ejecuta la celda de instalacion para desinstalarlo y luego reinicia el runtime antes de entrenar."
    )

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

print("Drive project dir:", DRIVE_PROJECT_DIR)

## 3. Configuración central

`smoke_free` está pensado para comprobar que todo corre. `pro` es el preset para entrenamiento más serio en Colab Pro.

In [ ]:
RUN_MODE = "smoke_free"  # "smoke_free" o "pro"
BASE_MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

PRESETS = {
    "smoke_free": {
        "max_train_images": 5,
        "max_train_steps": 30,
        "resolution": 512,
        "checkpointing_steps": 15,
        "comparison_prompts": 1,
        "inference_steps": 15,
        "guidance_scale": 6.5,
        "lora_rank": 8,
        "learning_rate": 1e-4,
        "train_batch_size": 1,
        "gradient_accumulation_steps": 4,
    },
    "pro": {
        "max_train_images": None,
        "max_train_steps": 350,
        "resolution": 768,
        "checkpointing_steps": 100,
        "comparison_prompts": 6,
        "inference_steps": 30,
        "guidance_scale": 7.0,
        "lora_rank": 16,
        "learning_rate": 1e-4,
        "train_batch_size": 1,
        "gradient_accumulation_steps": 4,
    },
}

if RUN_MODE not in PRESETS:
    raise ValueError(f"RUN_MODE invalido: {RUN_MODE}. Usa {list(PRESETS)}")

preset = PRESETS[RUN_MODE]
MAX_TRAIN_IMAGES = preset["max_train_images"]
MAX_TRAIN_STEPS = preset["max_train_steps"]
RESOLUTION = preset["resolution"]
CHECKPOINTING_STEPS = preset["checkpointing_steps"]
COMPARISON_PROMPTS = preset["comparison_prompts"]
INFERENCE_STEPS = preset["inference_steps"]
GUIDANCE_SCALE = preset["guidance_scale"]
VISUAL_LORA_RANK = preset["lora_rank"]
VISUAL_LEARNING_RATE = preset["learning_rate"]
TRAIN_BATCH_SIZE = preset["train_batch_size"]
GRADIENT_ACCUMULATION_STEPS = preset["gradient_accumulation_steps"]
MIXED_PRECISION = "fp16"

RUN_TRAINING = True              # Cambia a True cuando quieras entrenar.
RUN_BASELINE_GENERATION = True    # Genera SDXL base para comparar.
RUN_LORA_GENERATION = True        # Genera con LoRA si ya existe adapter.
RUN_EVALUATION = True             # Calcula metricas ligeras si hay imagenes.
RUN_FID_KID = True               # Opcional y recomendado solo en modo pro con muchas muestras.

VISUAL_DEALERSHIP_NAME = "AUTOESPAR"
VISUAL_PRIMARY_BRAND = "TOYOTA"
VISUAL_MODEL_SERIES = "Toyota Corolla Cross Hybrid Electric 2025"
VISUAL_CONTACT_PHONE = "971 210 520"
VISUAL_TRIGGER_WORD = VISUAL_DEALERSHIP_NAME
VISUAL_LEGAL_BAR_GUIDANCE = (
    "bottom legal conditions strip, usually a black, white, or red horizontal rectangle "
    "with very small Spanish legal disclaimer text"
)
INSTANCE_PROMPT = (
    f"{VISUAL_TRIGGER_WORD} automotive dealership marketing campaign asset for {VISUAL_PRIMARY_BRAND}, "
    f"commercial car banner layout, {VISUAL_LEGAL_BAR_GUIDANCE}"
)
NEGATIVE_PROMPT = (
    "blurry, low quality, watermark, deformed car, extra wheels, broken headlights, bad perspective, "
    "jpeg artifacts, distorted people, malformed logo, misspelled AUTOESPAR, misspelled TOYOTA, "
    "wrong phone number, unreadable CTA"
)

print(json.dumps({
    "run_mode": RUN_MODE,
    "base_model": BASE_MODEL_ID,
    "max_train_images": MAX_TRAIN_IMAGES,
    "max_train_steps": MAX_TRAIN_STEPS,
    "resolution": RESOLUTION,
    "checkpointing_steps": CHECKPOINTING_STEPS,
    "run_training": RUN_TRAINING,
    "run_baseline_generation": RUN_BASELINE_GENERATION,
    "run_lora_generation": RUN_LORA_GENERATION,
    "run_evaluation": RUN_EVALUATION,
}, indent=2))

## 4. Rutas de proyecto, Drive e input visual

El notebook puede leer el dataset visual desde Drive o desde el repo clonado. Por defecto prioriza Drive:

```text
/content/drive/MyDrive/dmc-tp3-gen-multimodal/diffusion_lora/data/car_campaign_lora/
```

Si esa carpeta no existe o no tiene metadata, usa el dataset del repo:

```text
/content/dmc-tp3-gen-multimodal/data/car_campaign_lora/
```

Los artefactos pesados siempre salen a Drive.


In [ ]:
CANDIDATE_ROOTS = [
    Path.cwd(),
    Path("/content/dmc-tp3-gen-multimodal"),
    Path("/kaggle/working/dmc-tp3-gen-multimodal"),
]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "data" / "car_campaign_lora").exists()), Path.cwd())
DATA_ROOT = PROJECT_ROOT / "data"
LOCAL_VISUAL_DATA_DIR = DATA_ROOT / "car_campaign_lora"

DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "data"
DRIVE_VISUAL_DATA_DIR = DRIVE_DATA_DIR / "car_campaign_lora"


def visual_data_dir_has_metadata(path: Path):
    return (path / "metadata.csv").exists() or (path / "metadata_template.csv").exists()


if visual_data_dir_has_metadata(DRIVE_VISUAL_DATA_DIR):
    VISUAL_DATA_DIR = DRIVE_VISUAL_DATA_DIR
    INPUT_SOURCE = "drive"
elif visual_data_dir_has_metadata(LOCAL_VISUAL_DATA_DIR):
    VISUAL_DATA_DIR = LOCAL_VISUAL_DATA_DIR
    INPUT_SOURCE = "repo"
else:
    VISUAL_DATA_DIR = DRIVE_VISUAL_DATA_DIR
    INPUT_SOURCE = "drive_missing"

VISUAL_METADATA_PATH = VISUAL_DATA_DIR / "metadata.csv"
VISUAL_METADATA_TEMPLATE_PATH = VISUAL_DATA_DIR / "metadata_template.csv"

PREPARED_DATASET_DIR = DRIVE_PROJECT_DIR / "prepared_dataset"
TRAINING_OUTPUT_DIR = DRIVE_PROJECT_DIR / "training_output"
CHECKPOINTS_DIR = DRIVE_PROJECT_DIR / "checkpoints"
LORA_OUTPUT_DIR = DRIVE_PROJECT_DIR / "lora_adapter"
GENERATED_DIR = DRIVE_PROJECT_DIR / "generated"
BASE_GENERATED_DIR = GENERATED_DIR / "base"
LORA_GENERATED_DIR = GENERATED_DIR / "lora"
COMPARISON_CANVAS_DIR = GENERATED_DIR / "comparison_canvases"
EVALUATION_DIR = DRIVE_PROJECT_DIR / "evaluation"
LOGS_DIR = DRIVE_PROJECT_DIR / "logs"
SCRIPT_PATH = DRIVE_PROJECT_DIR / "train_dreambooth_lora_sdxl.py"

for path in [DRIVE_DATA_DIR, DRIVE_VISUAL_DATA_DIR, PREPARED_DATASET_DIR, TRAINING_OUTPUT_DIR, CHECKPOINTS_DIR, LORA_OUTPUT_DIR, GENERATED_DIR, BASE_GENERATED_DIR, LORA_GENERATED_DIR, COMPARISON_CANVAS_DIR, EVALUATION_DIR, LOGS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input source:", INPUT_SOURCE)
print("Local visual data dir:", LOCAL_VISUAL_DATA_DIR)
print("Drive visual data dir:", DRIVE_VISUAL_DATA_DIR)
print("Visual data dir selected:", VISUAL_DATA_DIR)
print("Metadata path:", VISUAL_METADATA_PATH)
print("Metadata template:", VISUAL_METADATA_TEMPLATE_PATH)
print("Training output dir:", TRAINING_OUTPUT_DIR)
print("Checkpoints dir:", CHECKPOINTS_DIR)
print("LoRA output dir:", LORA_OUTPUT_DIR)
print("Evaluation dir:", EVALUATION_DIR)


## 5. Cargar y validar metadata visual

Usa `metadata.csv` si existe; de lo contrario usa `metadata_template.csv`. El CSV debe tener `file_path` y `caption`.

Si quieres trabajar 100% desde Drive, coloca los archivos aquí:

```text
{DRIVE_PROJECT_DIR}/data/car_campaign_lora/metadata.csv
{DRIVE_PROJECT_DIR}/data/car_campaign_lora/images/
```

Los `file_path` relativos del CSV se resuelven contra la carpeta donde esté ese CSV.


In [ ]:
SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}
VISUAL_GUIDANCE_TERMS = [
    VISUAL_DEALERSHIP_NAME,
    VISUAL_PRIMARY_BRAND,
    "Toyota",
    VISUAL_MODEL_SERIES,
    "Ford",
    "Mustang",
]


def has_any_guidance_term(caption: str, terms):
    caption_lower = caption.lower()
    return any(term.lower() in caption_lower for term in terms if term)


def resolve_metadata_path():
    if VISUAL_METADATA_PATH.exists():
        return VISUAL_METADATA_PATH
    if VISUAL_METADATA_TEMPLATE_PATH.exists():
        print(f"No existe {VISUAL_METADATA_PATH}. Se usara template preparado: {VISUAL_METADATA_TEMPLATE_PATH}")
        return VISUAL_METADATA_TEMPLATE_PATH
    raise FileNotFoundError("No se encontro metadata.csv ni metadata_template.csv en data/car_campaign_lora/.")


def load_visual_metadata(metadata_path: Path, guidance_terms):
    df = pd.read_csv(metadata_path)
    required = {"file_path", "caption"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas en metadata visual: {sorted(missing)}")

    records = []
    errors = []
    root = metadata_path.parent
    for idx, row in df.iterrows():
        raw_path = Path(str(row["file_path"]))
        image_path = raw_path if raw_path.is_absolute() else (root / raw_path).resolve()
        caption = str(row["caption"]).strip()
        caption_lower = caption.lower()

        if image_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            errors.append((idx, f"extension no soportada: {image_path}"))
        if not image_path.exists():
            errors.append((idx, f"imagen no existe: {image_path}"))
        if not caption:
            errors.append((idx, "caption vacia"))
        if caption and not has_any_guidance_term(caption, guidance_terms):
            errors.append((idx, f"caption debe mencionar al menos una guia visual: {guidance_terms}"))

        records.append({
            "source_index": int(idx),
            "image": image_path,
            "caption": caption,
            "has_dealership": VISUAL_DEALERSHIP_NAME.lower() in caption_lower,
            "has_primary_brand": VISUAL_PRIMARY_BRAND.lower() in caption_lower or "toyota" in caption_lower,
            "has_model_series": VISUAL_MODEL_SERIES.lower() in caption_lower,
            "has_legal_bar": "conditions/legal disclaimer" in caption_lower or "legal conditions" in caption_lower,
            "has_phone": "cotiza al" in caption_lower or any(ch.isdigit() for ch in caption),
        })

    if errors:
        print("Errores de metadata visual, mostrando maximo 15:")
        for err in errors[:15]:
            print(err)
        raise ValueError(f"Metadata visual invalida: {len(errors)} errores.")

    if MAX_TRAIN_IMAGES:
        records = records[:MAX_TRAIN_IMAGES]

    summary = pd.DataFrame(records)
    print(f"Metadata visual usada: {metadata_path}")
    print(f"Imagenes seleccionadas para este RUN_MODE: {len(records)}")
    print("Filas con dealership AUTOESPAR:", int(summary["has_dealership"].sum()))
    print("Filas con marca TOYOTA:", int(summary["has_primary_brand"].sum()))
    print("Filas con modelo/serie ejemplo:", int(summary["has_model_series"].sum()))
    print("Filas con franja de condiciones:", int(summary["has_legal_bar"].sum()))
    print("Filas con telefono o CTA numerico:", int(summary["has_phone"].sum()))
    return records, summary

metadata_used_path = resolve_metadata_path()
visual_records, visual_summary = load_visual_metadata(metadata_used_path, VISUAL_GUIDANCE_TERMS)

preview_rows = []
for record in visual_records[:5]:
    preview_rows.append({
        "image": record["image"].name,
        "caption": record["caption"][:220],
        "has_dealership": record["has_dealership"],
        "has_primary_brand": record["has_primary_brand"],
        "has_legal_bar": record["has_legal_bar"],
    })
    display(ImageOps.contain(Image.open(record["image"]).convert("RGB"), (320, 220)))

display(pd.DataFrame(preview_rows))

## 6. Preparar dataset DreamBooth en Drive

Copia las imágenes seleccionadas como PNG RGB a Drive y guarda el resumen auditable del dataset.

In [ ]:
if PREPARED_DATASET_DIR.exists():
    shutil.rmtree(PREPARED_DATASET_DIR)
PREPARED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

safe_prefix = re.sub(r"[^a-z0-9]+", "_", VISUAL_TRIGGER_WORD.lower()).strip("_")
dataset_summary = []

for idx, record in enumerate(visual_records, start=1):
    src = record["image"]
    dst = PREPARED_DATASET_DIR / f"{safe_prefix}_{idx:03d}.png"
    image = Image.open(src).convert("RGB")
    image.save(dst)
    dataset_summary.append({
        "file_name": dst.name,
        "prepared_path": str(dst),
        "source_path": str(src),
        "caption": record["caption"],
        "has_dealership": record["has_dealership"],
        "has_primary_brand": record["has_primary_brand"],
        "has_model_series": record["has_model_series"],
        "has_legal_bar": record["has_legal_bar"],
        "has_phone": record["has_phone"],
    })

summary_df = pd.DataFrame(dataset_summary)
summary_path = EVALUATION_DIR / "visual_training_dataset_summary.csv"
summary_df.to_csv(summary_path, index=False)

# Copia de metadata usada para trazabilidad. Si el input ya viene de Drive, evita copiar sobre si mismo.
metadata_trace_path = DRIVE_DATA_DIR / metadata_used_path.name
if metadata_used_path.resolve() != metadata_trace_path.resolve():
    shutil.copy2(metadata_used_path, metadata_trace_path)

print("Dataset DreamBooth preparado en:", PREPARED_DATASET_DIR)
print("Instance prompt:", INSTANCE_PROMPT)
print("Resumen guardado en:", summary_path)
display(summary_df.head())

## 7. Descargar script oficial DreamBooth LoRA SDXL

Se guarda en Drive para que la sesión pueda reusarlo. Si quieres fijar una versión exacta, reemplaza `main` por un commit de Diffusers.

In [ ]:
if not SCRIPT_PATH.exists():
    url = "https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_sdxl.py"
    subprocess.run(["wget", "-q", url, "-O", str(SCRIPT_PATH)], check=True)

if not SCRIPT_PATH.exists() or SCRIPT_PATH.stat().st_size == 0:
    raise RuntimeError("No se pudo descargar train_dreambooth_lora_sdxl.py")

print("Script DreamBooth:", SCRIPT_PATH)
print("Tamano bytes:", SCRIPT_PATH.stat().st_size)


## 8. Entrenar LoRA visual

Por seguridad `RUN_TRAINING=False` al inicio. Actívalo cuando hayas validado metadata, Drive y GPU. En modo `smoke_free` el entrenamiento es solo una prueba funcional.

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

training_config = {
    "base_model": BASE_MODEL_ID,
    "instance_data_dir": str(PREPARED_DATASET_DIR),
    "training_output_dir": str(TRAINING_OUTPUT_DIR),
    "lora_output_dir": str(LORA_OUTPUT_DIR),
    "checkpoints_dir": str(CHECKPOINTS_DIR),
    "instance_prompt": INSTANCE_PROMPT,
    "resolution": RESOLUTION,
    "max_train_steps": MAX_TRAIN_STEPS,
    "checkpointing_steps": CHECKPOINTING_STEPS,
    "rank": VISUAL_LORA_RANK,
    "learning_rate": VISUAL_LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "mixed_precision": MIXED_PRECISION,
    "seed": SEED,
    "run_mode": RUN_MODE,
}

training_config_path = EVALUATION_DIR / "training_config.json"
training_config_path.write_text(json.dumps(training_config, indent=2), encoding="utf-8")
print(json.dumps(training_config, indent=2))
print("Config guardada en:", training_config_path)

In [ ]:
if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Se requiere GPU para entrenar DreamBooth LoRA visual.")
    if not any(PREPARED_DATASET_DIR.glob("*.png")):
        raise RuntimeError("No hay imagenes preparadas para entrenamiento.")
    torchao_version = get_installed_package_version("torchao")
    if torchao_version and tuple(int(part) for part in torchao_version.split(".")[:2]) < (0, 16):
        raise RuntimeError(
            "Entrenamiento bloqueado por torchao incompatible (" + torchao_version + "). "
            "Vuelve a ejecutar la celda de dependencias, reinicia el runtime y prueba otra vez."
        )

    cmd = [
        "accelerate", "launch", str(SCRIPT_PATH),
        f"--pretrained_model_name_or_path={BASE_MODEL_ID}",
        f"--instance_data_dir={PREPARED_DATASET_DIR}",
        f"--output_dir={TRAINING_OUTPUT_DIR}",
        f"--instance_prompt={INSTANCE_PROMPT}",
        f"--resolution={RESOLUTION}",
        f"--train_batch_size={TRAIN_BATCH_SIZE}",
        f"--gradient_accumulation_steps={GRADIENT_ACCUMULATION_STEPS}",
        f"--learning_rate={VISUAL_LEARNING_RATE}",
        "--lr_scheduler=constant",
        "--lr_warmup_steps=0",
        f"--max_train_steps={MAX_TRAIN_STEPS}",
        f"--checkpointing_steps={CHECKPOINTING_STEPS}",
        f"--checkpoints_total_limit=3",
        f"--rank={VISUAL_LORA_RANK}",
        f"--seed={SEED}",
        f"--mixed_precision={MIXED_PRECISION}",
        "--gradient_checkpointing",
    ]
    print("Ejecutando:")
    print(" ".join(cmd))
    start = time.perf_counter()
    result = subprocess.run(cmd, text=True, capture_output=True)
    elapsed = time.perf_counter() - start

    (LOGS_DIR / "train_stdout.log").write_text(result.stdout, encoding="utf-8")
    (LOGS_DIR / "train_stderr.log").write_text(result.stderr, encoding="utf-8")
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
        if "Found an incompatible version of torchao" in result.stdout or "Found an incompatible version of torchao" in result.stderr:
            raise RuntimeError(
                "El entrenamiento fallo por torchao incompatible en Colab. "
                "Ejecuta la celda de dependencias actualizada, reinicia el runtime y vuelve a correr desde imports."
            )
        raise RuntimeError("El entrenamiento LoRA visual fallo. Revisa logs en Drive.")
    print(f"Entrenamiento terminado en {elapsed / 60:.2f} min")
    # Diffusers guarda el adapter final y los checkpoint-* dentro de output_dir.
    # Copiamos cada artefacto a las carpetas estables esperadas por el resto del notebook.
    LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
    copied_adapters = []
    for adapter_file in TRAINING_OUTPUT_DIR.glob("*.safetensors"):
        target = LORA_OUTPUT_DIR / adapter_file.name
        shutil.copy2(adapter_file, target)
        copied_adapters.append(str(target))
    for checkpoint_dir in TRAINING_OUTPUT_DIR.glob("checkpoint-*"):
        if checkpoint_dir.is_dir():
            target = CHECKPOINTS_DIR / checkpoint_dir.name
            shutil.copytree(checkpoint_dir, target, dirs_exist_ok=True)
    print("Adapters copiados a lora_adapter:", copied_adapters)
    print("Checkpoints disponibles:", [p.name for p in sorted(CHECKPOINTS_DIR.glob("checkpoint-*"))])
else:
    print("RUN_TRAINING=False. Se omite entrenamiento visual.")

lora_files = sorted(LORA_OUTPUT_DIR.rglob("*.safetensors"))
print("LoRA visual files:", [str(p.relative_to(LORA_OUTPUT_DIR)) for p in lora_files])

## 9. Prompts de comparación

Se usan los mismos prompts y seeds para el modelo base y el modelo con LoRA.

In [ ]:
PLACEMENTS = [
    {"name": "website_hero", "label": "website hero banner", "width": 768, "height": 432, "composition": "wide horizontal 16:9 layout with clean space for headline and CTA overlay"},
    {"name": "instagram_feed", "label": "Instagram feed ad", "width": 640, "height": 640, "composition": "square centered composition for social media feed"},
    {"name": "vertical_story", "label": "Instagram story or TikTok vertical ad", "width": 432, "height": 768, "composition": "vertical 9:16 mobile-first composition with dealership branding near the top"},
    {"name": "showroom_poster", "label": "dealership showroom poster", "width": 640, "height": 832, "composition": "premium vertical print poster composition with vehicle hero image"},
    {"name": "highway_billboard", "label": "highway billboard", "width": 896, "height": 384, "composition": "ultra-wide billboard composition readable from distance with short CTA area"},
    {"name": "email_header", "label": "email campaign header", "width": 768, "height": 320, "composition": "clean wide email header composition with space for offer copy"},
]

BASE_VISUAL_PROMPT = (
    f"{VISUAL_TRIGGER_WORD} automotive dealership marketing banner for {VISUAL_PRIMARY_BRAND} {VISUAL_MODEL_SERIES}, "
    f"campaign asset with dealership logo area, TOYOTA brand badge area, CTA area reading cotiza al {VISUAL_CONTACT_PHONE}, "
    f"{VISUAL_LEGAL_BAR_GUIDANCE}, premium energetic dealership retail campaign, polished automotive reflections, "
    "high quality commercial advertising, crisp vehicle proportions"
)


def build_visual_prompt(placement):
    return (
        f"{BASE_VISUAL_PROMPT}, {placement['label']}, {placement['composition']}, "
        f"visible {VISUAL_DEALERSHIP_NAME} dealership identity, visible {VISUAL_PRIMARY_BRAND} brand presence, "
        f"optional model label for {VISUAL_MODEL_SERIES}, professional campaign layout"
    )

prompts_df = pd.DataFrame([
    {
        "name": placement["name"],
        "label": placement["label"],
        "width": int(placement["width"]),
        "height": int(placement["height"]),
        "prompt": build_visual_prompt(placement),
        "negative_prompt": NEGATIVE_PROMPT,
        "seed": SEED + idx,
    }
    for idx, placement in enumerate(PLACEMENTS[:COMPARISON_PROMPTS])
])

prompts_path = EVALUATION_DIR / "comparison_prompts.csv"
prompts_df.to_csv(prompts_path, index=False)
print("Prompts guardados en:", prompts_path)
display(prompts_df[["name", "width", "height", "seed", "prompt"]])

## 10. Cargar pipeline SDXL

La misma instancia de pipeline se usa para base y LoRA, cargando o descargando pesos LoRA según corresponda.

In [ ]:
def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_sdxl_pipeline():
    from diffusers import DPMSolverMultistepScheduler, StableDiffusionXLPipeline

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    pipe = StableDiffusionXLPipeline.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=dtype,
        use_safetensors=True,
        variant="fp16" if torch.cuda.is_available() else None,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception as exc:
        print("xformers no disponible o no compatible:", exc)
    return pipe

should_load_pipe = RUN_BASELINE_GENERATION or RUN_LORA_GENERATION
pipe = load_sdxl_pipeline() if should_load_pipe else None
print("Pipeline cargado:", pipe is not None)

## 11. Generar base vs LoRA

Guarda PNGs por modelo, canvases lado a lado y `generation_records.json`.

In [ ]:
def has_lora_adapter():
    return any(LORA_OUTPUT_DIR.rglob("*.safetensors"))


def set_lora_state(pipe, use_lora: bool):
    if use_lora:
        if not has_lora_adapter():
            raise RuntimeError(f"No se encontro adapter LoRA en {LORA_OUTPUT_DIR}")
        try:
            pipe.load_lora_weights(str(LORA_OUTPUT_DIR))
        except Exception as exc:
            print("Aviso al cargar LoRA, se intentara continuar:", exc)
    else:
        try:
            pipe.unload_lora_weights()
        except Exception:
            pass


def generate_one(row, variant):
    if pipe is None:
        raise RuntimeError("Carga primero el pipeline SDXL.")
    use_lora = variant == "lora"
    set_lora_state(pipe, use_lora)

    out_dir = LORA_GENERATED_DIR if use_lora else BASE_GENERATED_DIR
    generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(int(row["seed"]))
    start = time.perf_counter()
    image = pipe(
        prompt=row["prompt"],
        negative_prompt=row["negative_prompt"],
        width=int(row["width"]),
        height=int(row["height"]),
        num_inference_steps=INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    ).images[0]
    latency = time.perf_counter() - start
    image_path = out_dir / f"{row['name']}_{variant}_seed_{int(row['seed'])}.png"
    image.save(image_path)
    return image, image_path, latency


def save_comparison_canvas(base_path, lora_path, placement, seed):
    base_image = Image.open(base_path).convert("RGB")
    lora_image = Image.open(lora_path).convert("RGB")
    canvas = Image.new("RGB", (base_image.width + lora_image.width, max(base_image.height, lora_image.height)), "white")
    canvas.paste(base_image, (0, 0))
    canvas.paste(lora_image, (base_image.width, 0))
    canvas_path = COMPARISON_CANVAS_DIR / f"{placement}_base_vs_lora_seed_{seed}.png"
    canvas.save(canvas_path)
    return canvas_path

records = []
errors = []

for _, row in prompts_df.iterrows():
    row = row.to_dict()
    if RUN_BASELINE_GENERATION:
        try:
            image, path, latency = generate_one(row, "base")
            records.append({
                "variant": "base",
                "placement": row["name"],
                "path": str(path),
                "prompt": row["prompt"],
                "negative_prompt": row["negative_prompt"],
                "seed": int(row["seed"]),
                "width": int(row["width"]),
                "height": int(row["height"]),
                "model": BASE_MODEL_ID,
                "lora_dir": None,
                "latency_sec": latency,
                "error": None,
            })
            print("base", row["name"], path, f"{latency:.2f}s")
            display(image)
        except Exception as exc:
            errors.append({"variant": "base", "placement": row["name"], "error": str(exc)})
            print("Error base", row["name"], exc)

    if RUN_LORA_GENERATION:
        if has_lora_adapter():
            try:
                image, path, latency = generate_one(row, "lora")
                records.append({
                    "variant": "lora",
                    "placement": row["name"],
                    "path": str(path),
                    "prompt": row["prompt"],
                    "negative_prompt": row["negative_prompt"],
                    "seed": int(row["seed"]),
                    "width": int(row["width"]),
                    "height": int(row["height"]),
                    "model": BASE_MODEL_ID,
                    "lora_dir": str(LORA_OUTPUT_DIR),
                    "latency_sec": latency,
                    "error": None,
                })
                print("lora", row["name"], path, f"{latency:.2f}s")
                display(image)
            except Exception as exc:
                errors.append({"variant": "lora", "placement": row["name"], "error": str(exc)})
                print("Error LoRA", row["name"], exc)
        else:
            print("RUN_LORA_GENERATION=True, pero aun no hay adapter LoRA. Se omite variante LoRA.")

records_path = EVALUATION_DIR / "generation_records.json"
records_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
errors_path = EVALUATION_DIR / "generation_errors.json"
errors_path.write_text(json.dumps(errors, ensure_ascii=False, indent=2), encoding="utf-8")

# Canvases para pares completos.
records_df = pd.DataFrame(records)
canvas_records = []
if not records_df.empty and {"base", "lora"}.issubset(set(records_df["variant"])):
    for placement, group in records_df.groupby("placement"):
        base_rows = group[group["variant"] == "base"]
        lora_rows = group[group["variant"] == "lora"]
        if not base_rows.empty and not lora_rows.empty:
            seed = int(base_rows.iloc[0]["seed"])
            canvas_path = save_comparison_canvas(base_rows.iloc[0]["path"], lora_rows.iloc[0]["path"], placement, seed)
            canvas_records.append({"placement": placement, "seed": seed, "canvas_path": str(canvas_path)})
            display(Image.open(canvas_path))

canvas_path = EVALUATION_DIR / "comparison_canvases.json"
canvas_path.write_text(json.dumps(canvas_records, ensure_ascii=False, indent=2), encoding="utf-8")
print("Generation records:", records_path)
print("Generation errors:", errors_path)
print("Canvas records:", canvas_path)
release_memory()

## 12. Métricas ligeras para reporte

Calcula latencia, CLIPScore texto-imagen, similitud de cada imagen contra el dataset de entrenamiento y diversidad intra-modelo. Si `torchmetrics` falla, usa embeddings CLIP de `transformers` directamente.

In [ ]:
def load_generation_records():
    path = EVALUATION_DIR / "generation_records.json"
    if not path.exists():
        return pd.DataFrame()
    data = json.loads(path.read_text(encoding="utf-8"))
    return pd.DataFrame(data)


def load_clip_model():
    from transformers import CLIPModel, CLIPProcessor

    clip_model_id = "openai/clip-vit-base-patch32"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPModel.from_pretrained(clip_model_id).to(device)
    processor = CLIPProcessor.from_pretrained(clip_model_id)
    model.eval()
    return model, processor, device


def move_batch_to_device(batch, device):
    return {key: value.to(device) if hasattr(value, "to") else value for key, value in batch.items()}


def l2_normalize(embeds):
    return embeds / embeds.norm(dim=-1, keepdim=True)


def clip_text_image_scores(model, processor, device, rows):
    scores = []
    with torch.inference_mode():
        for row in rows:
            image = Image.open(row["path"]).convert("RGB")
            inputs = processor(text=[row["prompt"]], images=[image], return_tensors="pt", padding=True, truncation=True)
            inputs = move_batch_to_device(inputs, device)
            outputs = model(**inputs)
            image_embeds = l2_normalize(outputs.image_embeds)
            text_embeds = l2_normalize(outputs.text_embeds)
            score = float((image_embeds * text_embeds).sum(dim=-1).item())
            scores.append(score)
    return scores


def encode_images_clip(model, processor, device, image_paths, batch_size=8):
    embeddings = []
    with torch.inference_mode():
        for start in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[start:start + batch_size]
            images = [Image.open(path).convert("RGB") for path in batch_paths]
            inputs = processor(images=images, return_tensors="pt")
            inputs = move_batch_to_device(inputs, device)
            image_embeds = model.get_image_features(pixel_values=inputs["pixel_values"])
            if hasattr(image_embeds, "image_embeds"):
                image_embeds = image_embeds.image_embeds
            elif hasattr(image_embeds, "pooler_output"):
                image_embeds = image_embeds.pooler_output
            image_embeds = l2_normalize(image_embeds)
            embeddings.append(image_embeds.detach().cpu())
    if not embeddings:
        return torch.empty((0, 512))
    return torch.cat(embeddings, dim=0)


def mean_max_similarity_to_dataset(generated_embeds, dataset_embeds):
    if generated_embeds.numel() == 0 or dataset_embeds.numel() == 0:
        return []
    sims = generated_embeds @ dataset_embeds.T
    return sims.max(dim=1).values.numpy().tolist()


def mean_pairwise_similarity(embeds):
    if embeds.shape[0] < 2:
        return None
    sims = embeds @ embeds.T
    mask = ~torch.eye(sims.shape[0], dtype=torch.bool)
    return float(sims[mask].mean().item())

if RUN_EVALUATION:
    generation_df = load_generation_records()
    if generation_df.empty:
        print("No hay generation_records.json con imagenes generadas. Se omiten metricas.")
    else:
        existing_mask = generation_df["path"].map(lambda p: Path(p).exists())
        missing_count = int((~existing_mask).sum())
        generation_df = generation_df[existing_mask].copy()

        clip_model, clip_processor, clip_device = load_clip_model()
        row_dicts = generation_df.to_dict("records")
        generation_df["clip_text_image_similarity"] = clip_text_image_scores(clip_model, clip_processor, clip_device, row_dicts)

        dataset_paths = [str(p) for p in sorted(PREPARED_DATASET_DIR.glob("*.png"))]
        dataset_embeds = encode_images_clip(clip_model, clip_processor, clip_device, dataset_paths)
        generated_embeds = encode_images_clip(clip_model, clip_processor, clip_device, generation_df["path"].tolist())
        generation_df["max_clip_image_similarity_to_training_set"] = mean_max_similarity_to_dataset(generated_embeds, dataset_embeds)

        diversity_rows = []
        for variant, group in generation_df.groupby("variant"):
            embeds = encode_images_clip(clip_model, clip_processor, clip_device, group["path"].tolist())
            mean_sim = mean_pairwise_similarity(embeds)
            diversity_rows.append({
                "variant": variant,
                "mean_pairwise_clip_similarity": mean_sim,
                "diversity_score_1_minus_similarity": None if mean_sim is None else 1.0 - mean_sim,
                "image_count": int(len(group)),
            })
        diversity_df = pd.DataFrame(diversity_rows)

        manual_cols = {
            "brand_consistency_score": "",
            "layout_quality_score": "",
            "vehicle_quality_score": "",
            "legal_bar_present": "",
            "notes": "",
        }
        for col, default in manual_cols.items():
            generation_df[col] = default

        metrics_path = EVALUATION_DIR / "base_vs_lora_metrics.csv"
        generation_df.to_csv(metrics_path, index=False)

        summary_rows = []
        for variant, group in generation_df.groupby("variant"):
            summary_rows.append({
                "variant": variant,
                "image_count": int(len(group)),
                "missing_image_count": missing_count,
                "mean_latency_sec": float(group["latency_sec"].mean()),
                "mean_clip_text_image_similarity": float(group["clip_text_image_similarity"].mean()),
                "mean_max_clip_image_similarity_to_training_set": float(group["max_clip_image_similarity_to_training_set"].mean()),
            })
        summary_df = pd.DataFrame(summary_rows).merge(diversity_df, on=["variant", "image_count"], how="left")
        summary_path = EVALUATION_DIR / "base_vs_lora_metrics_summary.json"
        summary_path.write_text(json.dumps(summary_df.to_dict("records"), ensure_ascii=False, indent=2), encoding="utf-8")

        comparison_df = generation_df.pivot_table(
            index=["placement", "seed", "prompt"],
            columns="variant",
            values=["path", "latency_sec", "clip_text_image_similarity", "max_clip_image_similarity_to_training_set"],
            aggfunc="first",
        )
        comparison_df.columns = ["_".join([str(part) for part in col if part]) for col in comparison_df.columns]
        comparison_df = comparison_df.reset_index()
        comparison_csv_path = EVALUATION_DIR / "base_vs_lora_comparison.csv"
        comparison_df.to_csv(comparison_csv_path, index=False)

        print("Metricas guardadas en:", metrics_path)
        print("Resumen guardado en:", summary_path)
        print("Comparacion guardada en:", comparison_csv_path)
        display(generation_df)
        display(summary_df)

        del clip_model
        release_memory()
else:
    print("RUN_EVALUATION=False. Se omiten metricas.")

## 13. FID/KID opcional

FID/KID puede ser útil con más muestras, pero no es buena evidencia con pocas imágenes. Déjalo apagado para `smoke_free`.

In [ ]:
if RUN_FID_KID:
    if RUN_MODE != "pro":
        print("Aviso: FID/KID con pocas imagenes no es confiable. Considera usar RUN_MODE='pro'.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-fidelity>=0.3.0"], check=True)

    base_count = len(list(BASE_GENERATED_DIR.glob("*.png")))
    lora_count = len(list(LORA_GENERATED_DIR.glob("*.png")))
    if base_count < 10 or lora_count < 10:
        print(f"Muy pocas imagenes para FID/KID confiable: base={base_count}, lora={lora_count}")
    else:
        from torch_fidelity import calculate_metrics

        fid_kid_metrics = calculate_metrics(
            input1=str(BASE_GENERATED_DIR),
            input2=str(LORA_GENERATED_DIR),
            cuda=torch.cuda.is_available(),
            isc=False,
            fid=True,
            kid=True,
            verbose=True,
        )
        fid_kid_path = EVALUATION_DIR / "fid_kid_optional.json"
        fid_kid_path.write_text(json.dumps(fid_kid_metrics, indent=2), encoding="utf-8")
        print("FID/KID guardado en:", fid_kid_path)
        print(fid_kid_metrics)
else:
    print("RUN_FID_KID=False. Se omite FID/KID opcional.")

## 14. Checklist final

Al terminar una corrida completa, revisa que existan estos artefactos en Drive:

```text
evaluation/visual_training_dataset_summary.csv
evaluation/generation_records.json
evaluation/base_vs_lora_comparison.csv
evaluation/base_vs_lora_metrics.csv
evaluation/base_vs_lora_metrics_summary.json
generated/base/*.png
generated/lora/*.png
generated/comparison_canvases/*.png
lora_adapter/*.safetensors
checkpoints/
```

Para el futuro notebook de reporte, usa `base_vs_lora_metrics.csv`, `base_vs_lora_metrics_summary.json`, los canvases y la tabla manual de revisión.